# Train two-tower + cross-attention classifier - StereoQueerEval Task A

Binary stereotype detection (`yes` / `no`) with `src/train_xattn.py`.

The comment is the **query**; the video (title + description) is the
**key/value memory** - exactly the cross-attention of a Transformer decoder,
without any generation. Pool the comment side and read a 2-way logit.

- Encoder: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384-d,
  swap with `--backbone jhu-clsp/mmbert-base` for 768-d)
- Same video-level split / labels / metrics as `train.py`, so runs are comparable
- One encoder is shared by both towers, so the video K/V can be cached per video

Works on **Kaggle** and Colab.


In [ ]:
import os, subprocess

if not os.path.isdir('Try'):
    subprocess.run('git clone https://github.com/AmnO-O/Try.git', shell=True, check=True)
os.chdir('Try')
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
print('cwd:', os.getcwd())
print('src has train_xattn.py:', os.path.exists('src/train_xattn.py'))


## 1. Locate the data

The TSVs are not committed. This cell finds them automatically (Kaggle input,
`/content`, cwd, `./LGBT`) and exports `LGBT_DATA_DIR`.


In [ ]:
import glob, os

candidates = []
if 'LGBT_DATA_DIR' in os.environ and os.path.isdir(os.environ['LGBT_DATA_DIR']):
    candidates.append(os.environ['LGBT_DATA_DIR'])
if 'KAGGLE_INPUT_PATH' in os.environ and os.path.isdir(os.environ['KAGGLE_INPUT_PATH']):
    candidates.append(os.environ['KAGGLE_INPUT_PATH'])
for root in ('/kaggle/input', '/content', os.path.abspath('.')):
    candidates.append(root)
if os.path.isdir('LGBT'):
    candidates.append(os.path.abspath('LGBT'))

def _first(roots):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', 'StereoQueerEval_*_training.tsv'), recursive=True))
        if hits:
            return hits
    return []

hits = _first(candidates)
if not hits:
    raise RuntimeError('StereoQueerEval_*_training.tsv not found - mount the dataset first')
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs(os.environ['LGBT_WEIGHTS_DIR'], exist_ok=True)
print('data dir :', os.environ['LGBT_DATA_DIR'])
print('weights  :', os.environ['LGBT_WEIGHTS_DIR'])
print('files    :', [os.path.basename(p) for p in hits])
assert len(hits) >= 1


## 2. Sanity check the split

Row counts, label balance and the video-grouped train/val split. Defaults are
`val-fraction=0.1`, `split-seed=42` - the same ones the cells below use.


In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod
from config import MAX_LEN

df = data_mod.load_frame(max_len=MAX_LEN)
print('rows:', len(df), '| by lang:', df['lang'].value_counts().to_dict())
tr, va = data_mod.split_by_video(df, 0.1, 42)
print('train/val:', len(tr), '/', len(va))
print('video overlap:', len(set(tr['yt_title']) & set(va['yt_title'])))


## 3. Train the cross-attention model

`comment` -> query tower, `title [SEP] description` -> key/value tower, 4 heads,
1 cross-attention layer, last 2 encoder blocks unfrozen. The script prints
per-epoch metrics and, at the end, the validation confusion matrix + per-language
macro-F1. History lands in `weights/best_xattn_stereotype_history.csv`.

Start here; the sweep cell in section 6 widens this afterwards.


In [ ]:
!python src/train_xattn.py --epochs 10 --batch-size 32 --max-len 128 --video-max-len 192 --unfreeze-layers 2 --heads 4 --layers 1


## 4. Loss and macro-F1 curves


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

HIST = os.path.join(os.environ['LGBT_WEIGHTS_DIR'], 'best_xattn_stereotype_history.csv')
h = pd.read_csv(HIST)
print(h.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(h['epoch'], h['train_loss'], '--', label='train')
ax[0].plot(h['epoch'], h['val_loss'], '-', label='val')
ax[1].plot(h['epoch'], h['val_macro_f1'], '-o', ms=3, label='val macro-F1')
ax[0].set_title('loss'); ax[1].set_title('validation macro-F1')
for a in ax:
    a.grid(alpha=0.3); a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.show()

best = h.loc[h['val_macro_f1'].idxmax()]
print(f"best epoch {int(best['epoch'])} | macro_f1 {best['val_macro_f1']:.4f} | acc {best['val_acc']:.4f}")


## 5. Confusion matrix + per-language (best checkpoint)

Reloads `weights/best_xattn_stereotype.pt` and scores the same video-grouped
validation split that training used.


In [ ]:
import numpy as np
import torch
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

from model_xattn import build_model, DEFAULT_BACKBONE
from model import load_tokenizer
from train import resolve_device
import train_xattn as tx

dev = resolve_device('cuda' if torch.cuda.is_available() else 'cpu')
tok = load_tokenizer(DEFAULT_BACKBONE)
net = build_model(DEFAULT_BACKBONE, unfreeze_layers=0, heads=4, layers=1)
net.load_state_dict(torch.load('weights/best_xattn_stereotype.pt', map_location=dev))
net.to(dev).eval()

va = va.reset_index(drop=True)
va_loader = tx.make_loader(va, tok, 32, 128, 192)
out = tx.collect(net, va_loader, dev)
m = tx.metrics_from(out['preds'], out['labels'])
print({k: round(v, 4) for k, v in m.items() if k != 'loss'})

cm = confusion_matrix(out['labels'], out['preds'], labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['no', 'yes'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['no', 'yes'])
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.title('XAttn - validation'); plt.tight_layout(); plt.show()

print('--- per language ---')
for lang in sorted(va['lang'].unique()):
    sub = va[va['lang'] == lang].reset_index(drop=True)
    o = tx.collect(net, tx.make_loader(sub, tok, 32, 128, 192), dev)
    mm = tx.metrics_from(o['preds'], o['labels'])
    print(f"  [{lang}] acc {mm['acc']:.4f} | macro_f1 {mm['macro_f1']:.4f}")


## 6. Config sweep - heads / layers / unfreeze depth

Trains a few variants on the same split. One seed first; widen later. Each run
writes `weights/xattn_<name>_history.csv`.


In [ ]:
import subprocess
import pandas as pd

SEED = 42
CONFIGS = {
    'h4_l1_u2': '--heads 4 --layers 1 --unfreeze-layers 2',
    'h8_l1_u2': '--heads 8 --layers 1 --unfreeze-layers 2',
    'h4_l2_u2': '--heads 4 --layers 2 --unfreeze-layers 2',
    'h4_l1_u0': '--heads 4 --layers 1 --unfreeze-layers 0',
}

rows = []
for name, flags in CONFIGS.items():
    out = f'weights/xattn_{name}.pt'
    cmd = f'python src/train_xattn.py --split-seed {SEED} --seed {SEED} --epochs 10 ' \
          f'--batch-size 32 --max-len 128 --video-max-len 192 {flags} --out {out}'
    print('RUN:', cmd)
    subprocess.run(cmd, shell=True, check=True)
    h = pd.read_csv(out.replace('.pt', '_history.csv'))
    b = h.loc[h['val_macro_f1'].idxmax()]
    rows.append({'variant': name, 'best_epoch': int(b['epoch']),
                 'val_macro_f1': round(float(b['val_macro_f1']), 4),
                 'val_acc': round(float(b['val_acc']), 4)})

print(pd.DataFrame(rows).to_string(index=False))


## 7. Compare to the prompt baseline (`src/train.py`)

Same seeds and split, but the mmBERT prompt classifier instead of the two-tower
model. This is the number the cross-attention model has to beat. Reference best
so far: **0.7158** (batch 32, no extras).


In [ ]:
import subprocess, glob
import pandas as pd

SEEDS = (40, 41, 42)
rows = []

for s in SEEDS:
    out = f'weights/baseline_s{s}.pt'
    cmd = (f'python src/train.py --languages EN IT NL --split-seed {s} --seed {s} '
           f'--epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 '
           f'--readout-init random --out {out}')
    print('RUN:', cmd)
    subprocess.run(cmd, shell=True, check=True)
    h = pd.read_csv(out.replace('.pt', '_history.csv'))
    rows.append({'condition': 'prompt_baseline', 'seed': s,
                 'val_macro_f1': round(float(h['val_macro_f1'].max()), 4)})

for name in ('h4_l1_u2',):
    f = f'weights/xattn_{name}_history.csv'
    if os.path.exists(f):
        h = pd.read_csv(f)
        rows.append({'condition': f'xattn_{name}', 'seed': 42,
                     'val_macro_f1': round(float(h['val_macro_f1'].max()), 4)})

print(pd.DataFrame(rows).to_string(index=False))


### Without a notebook

```bash
git clone https://github.com/AmnO-O/Try.git && cd Try
pip install -r requirements.txt
export LGBT_DATA_DIR=/path/to/tsvs            # or put them in ./LGBT/

# cross-attention model (MiniLM, 384-d)
python src/train_xattn.py --epochs 10 --batch-size 32 --head-lr 1e-3 --unfreeze-layers 2

# prompt baseline on the same split
python src/train.py --epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 --readout-init random

# swap the encoder / widen the cross-attention
python src/train_xattn.py --backbone jhu-clsp/mmbert-base --heads 8 --layers 2 --unfreeze-layers 2

# augmented train split (built with src/augment_roundtrip.py)
python src/train_xattn.py --train-extras aug_rt_IT_to_EN_training.tsv,aug_rt_NL_to_EN_training.tsv
```

Checkpoints and history CSVs land in `weights/`.
